# PPE violation detection with YOLO26 + pose verification

Detects hard hats, hi-vis vests and safety footwear, and their absence, then uses
a pose model to check the equipment is actually **on the body** before anything is
reported as a violation.

## What this notebook produces

A per-person compliance record for every frame, of the form:

| person | helmet | vest | boots |
|---|---|---|---|
| 1 | COMPLIANT | COMPLIANT | INDETERMINATE (legs out of frame) |
| 2 | VIOLATION | COMPLIANT | COMPLIANT |
| 3 | REVIEW (carried, not worn) | VIOLATION | COMPLIANT |

Not a pile of boxes. A safety report needs findings attached to people, and it
needs to distinguish *"not wearing it"* from *"we could not see"*.

## The two-stage design

```
                 ┌─ yolo26s.pt      (fine-tuned, 7 PPE classes) ─┐
   frame ────────┤                                              ├──► ppe_fusion ──► per-person verdicts
                 └─ yolo26s-pose.pt (stock, COCO-17 keypoints) ──┘
```

The detector answers *"is there an uncovered head in this picture"*. The pose
model answers *"where is worker 3's head"*. Neither alone answers *"is worker 3
wearing their hard hat"*, which is the only question a safety report cares about.

The fusion layer assigns each PPE box to a person skeleton and tests it against
the body region it claims to protect. All thresholds are multiples of the
subject's torso length, so one configuration works at 4 m and at 40 m.

## Order of work

| # | Stage | Notes |
|---|---|---|
| 1 | Environment probe | picks model scale and batch from the GPU it finds |
| 2 | Dataset build | `ppe_harvest.py`; read `AUDIT.md` before training |
| 3 | Dataset audit | class balance, **box sizes → choose `imgsz`** |
| 4 | Train | detector only; the pose model stays stock |
| 5 | Evaluate | negative-class recall at a chosen operating point, not mAP |
| 6 | Pose sanity check | |
| 7 | Fusion | per-person verdicts, single image then batch |
| 8 | Video | temporal voting — one noisy frame is not a violation |
| 9 | Export | ONNX / TensorRT |
| 10 | Fine-tune on your own footage | the step that decides whether this works |

## Running it in your Ultralytics container

```bash
docker run --gpus all -it --rm \
  --ipc=host --shm-size=8g \
  -v "$PWD":/workspace -w /workspace \
  -p 8888:8888 \
  ultralytics/ultralytics:latest \
  bash -lc "pip install -q jupyterlab kagglehub huggingface_hub && \
            jupyter lab --ip=0.0.0.0 --allow-root --no-browser"
```

`--ipc=host` and `--shm-size` are not optional. Without them the PyTorch
DataLoader workers run out of shared memory partway through the first epoch and
die with an error that does not mention shared memory.

## 1 · Environment probe

In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path

REPO = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO / "src"))

import torch, ultralytics
from ultralytics import YOLO

print(f"ultralytics {ultralytics.__version__}   torch {torch.__version__}")

from ppe_memory import probe, recommend, report, preflight

# HOST RAM, not just GPU VRAM. Getting this wrong is what killed the first run
# of this project: the training process starved the VM until Jupyter itself
# could not allocate a few hundred kilobytes, and the OOM killer took the kernel.
FACTS = probe()
REC   = recommend(FACTS, imgsz=640)
print(report(FACTS, REC))

SCALE  = REC.model_scale
BATCH  = REC.batch
DEVICE = 0 if FACTS.gpu_count else "cpu"
VRAM   = FACTS.vram_gb

ok, msg = preflight(FACTS, REC)
print(f"\npreflight: {msg}")
if not ok:
    print("\nFix the memory budget before going further - see TROUBLESHOOTING.md.")

## 2 · Configuration

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd, numpy as np

CFG = dict(
    # --- data ---
    data_root        = Path("/workspace/data/ppe_yolo26"),
    include_nc       = False,   # True pulls in SH17 (CC BY-NC) - taints the weights
    harvest_limit    = None,    # e.g. 400 for a quick end-to-end rehearsal

    # --- detector ---
    det_weights      = f"yolo26{SCALE}.pt",
    imgsz            = 640,     # revisited in section 3 from the box-size audit
    epochs           = 120,
    batch            = BATCH,
    workers          = REC.workers,   # derived from host RAM, NOT a fixed 8
    cache            = REC.cache,     # never "ram" on a constrained box
    device           = DEVICE,
    patience         = 25,
    run_project      = "runs/ppe",
    run_name         = "detect_v1",

    # --- pose (stock weights, not trained) ---
    pose_weights     = f"yolo26{SCALE}-pose.pt",

    # --- inference ---
    end2end          = True,    # YOLO26 NMS-free head; False = one-to-many + NMS

    # --- reporting policy: precision floors per violation class ---
    # A missed violation is a safety risk. A false violation is a credibility
    # risk, and credibility is what gets a system switched off. Footwear is the
    # weakest leg of the taxonomy, so it is held to a stricter floor and reported
    # as advisory until site data is in.
    precision_floor  = {"no-helmet": 0.70, "no-vest": 0.65, "no-boots": 0.75},
)

# ---- plotting -------------------------------------------------------------
SURFACE   = "#fcfcfb"
INK       = "#0b0b0b"
INK_SOFT  = "#52514e"
GRID      = "#dcdcd6"
BLUE      = "#2a78d6"    # positive / neutral series
RED       = "#d03b3b"    # violation series  (validated CVD-safe against BLUE)
SERIES3   = ["#2a78d6", "#eb6834", "#1baf7a"]        # helmet, vest, boots
STATUS    = {"COMPLIANT": "#0ca30c", "VIOLATION": "#d03b3b",
             "REVIEW": "#fab219", "INDETERMINATE": "#8a8a85"}

mpl.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK, "axes.titlesize": 12, "axes.titleweight": "600",
    "axes.titlelocation": "left", "axes.titlepad": 14,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.7,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": INK_SOFT, "ytick.color": INK_SOFT,
    "font.size": 10, "legend.frameon": False, "figure.dpi": 110,
})

def bar_labels(ax, bars, fmt="{:,.0f}", pad=3, horiz=False):
    '''Direct labels. Three of the chosen hues sit under 3:1 contrast on this
    surface, so labels are the relief, not a nicety.'''
    for b in bars:
        if horiz:
            v = b.get_width()
            ax.text(v + pad, b.get_y() + b.get_height()/2, fmt.format(v),
                    va="center", ha="left", fontsize=9, color=INK_SOFT)
        else:
            v = b.get_height()
            ax.text(b.get_x() + b.get_width()/2, v + pad, fmt.format(v),
                    ha="center", va="bottom", fontsize=9, color=INK_SOFT)

from ppe_taxonomy import CANONICAL_CLASSES, VIOLATION_CLASSES
print(f"{len(CANONICAL_CLASSES)} classes: {CANONICAL_CLASSES}")
print(f"violation classes: {VIOLATION_CLASSES}")

## 3 · Build the dataset

`ppe_harvest.py` downloads every registered source, converts YOLO / VOC / COCO
labels to one label set, deduplicates by difference hash, splits, and oversamples
the violation classes **in train only**.

Two things to have in place first:

- **Kaggle** — `KAGGLE_USERNAME` and `KAGGLE_KEY` in the environment, or
  `kaggle.json` at `~/.config/kaggle/`.
- **HuggingFace** — nothing for public datasets; `huggingface-cli login` if you
  add a gated one.

Non-commercial sources are excluded by default. SH17 is a strong source of
bare-head negatives and is `CC BY-NC-SA 4.0`; weights trained on it inherit that
restriction, which for client work is a problem you find out about late. Set
`include_nc=True` deliberately, for research only.

In [ ]:
cmd = [sys.executable, str(REPO / "src" / "ppe_harvest.py"),
       "--out", str(CFG["data_root"])]
if CFG["include_nc"]:
    cmd.append("--include-noncommercial")
if CFG["harvest_limit"]:
    cmd += ["--limit", str(CFG["harvest_limit"])]

DATA_YAML = CFG["data_root"] / "data.yaml"

if DATA_YAML.exists():
    print(f"Dataset already built at {CFG['data_root']}  (delete it to rebuild)")
else:
    print(" ".join(cmd), "\n")
    subprocess.run(cmd, check=True)

print("\n" + "="*70)
print((CFG["data_root"] / "AUDIT.md").read_text()[:4000])

### Read the audit before you train

Three things in `AUDIT.md` decide whether the run is worth starting:

1. **UNRECOGNISED class names.** Anything listed there was in the data and got
   thrown away. Add it to `ALIASES` or to the source's `overrides` and rebuild.
2. **Head / foot cancellation counts.** A `head` box becomes `no-helmet` only when
   no helmet box overlaps it. Near-zero cancellations mean the source annotated
   those classes exclusively. A large count means it did not, and the subtraction
   just stopped every helmeted worker being labelled a violation.
3. **Class balance.** If a violation class has only a few hundred instances, the
   model will learn it badly and no amount of threshold tuning will fix that.

## 4 · Dataset audit

In [ ]:
from ppe_eval import load_gt, box_size_report

GT = {sp: load_gt(CFG["data_root"] / "images" / sp,
                  CFG["data_root"] / "labels" / sp)
      for sp in ("train", "val", "test")}
for sp, g in GT.items():
    n_box = sum(len(v) for v in g.values())
    print(f"{sp:>5}: {len(g):>6,} images  {n_box:>8,} boxes")

In [ ]:
# --- class balance -------------------------------------------------------
counts = {sp: pd.Series([CANONICAL_CLASSES[c] for items in g.values() for c, _ in items]
                        ).value_counts() for sp, g in GT.items()}
bal = pd.DataFrame(counts).reindex(CANONICAL_CLASSES).fillna(0).astype(int)
bal["total"] = bal.sum(axis=1)
bal["share"] = (bal["total"] / bal["total"].sum()).map("{:.1%}".format)
display(bal)

fig, ax = plt.subplots(figsize=(8.4, 4.0))
labels = [c + ("\n(violation)" if c in VIOLATION_CLASSES else "") for c in bal.index]
colours = [RED if c in VIOLATION_CLASSES else BLUE for c in bal.index]
bars = ax.bar(labels, bal["total"], color=colours, width=0.62)
bar_labels(ax, bars, pad=max(bal["total"]) * 0.012)
ax.set_title("Instances per class, all splits")
ax.set_ylabel("boxes")
ax.set_ylim(0, max(bal["total"]) * 1.14)
ax.grid(axis="x", visible=False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=BLUE, label="equipment present"),
                   Patch(color=RED,  label="violation class")],
          loc="upper right", ncols=2)
plt.tight_layout(); plt.show()

viol = bal.loc[VIOLATION_CLASSES, "total"]
print(f"\nviolation instances: {viol.sum():,} "
      f"({viol.sum()/bal['total'].sum():.1%} of all boxes)")
for c, v in viol.items():
    verdict = "workable" if v >= 2000 else ("thin" if v >= 500 else "TOO FEW - expect poor recall")
    print(f"  {c:<10} {v:>7,}   {verdict}")

In [ ]:
# --- object size: this is what picks imgsz -------------------------------
sz = box_size_report(GT["train"], CANONICAL_CLASSES)
display(sz)

fig, ax = plt.subplots(figsize=(8.4, 3.8))
sz_s = sz.sort_values("median_px_at_640")
colours = [RED if c in VIOLATION_CLASSES else BLUE for c in sz_s["class"]]
bars = ax.barh(sz_s["class"], sz_s["median_px_at_640"], color=colours, height=0.6)
bar_labels(ax, bars, fmt="{:.0f} px", pad=0.6, horiz=True)
ax.axvline(20, color=INK_SOFT, lw=1.2, ls="--")
ax.text(20.8, -0.42, "20 px - below this, detection is unreliable",
        fontsize=9, color=INK_SOFT)
ax.set_title("Median object size when the frame is resized to 640 px")
ax.set_xlabel("median box side, pixels at imgsz=640")
ax.set_xlim(0, max(sz_s["median_px_at_640"]) * 1.22)
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

small = sz[sz["small_object"]]["class"].tolist()
if small:
    smallest = float(sz["median_px_at_640"].min())
    rec = 640
    while rec < 1280 and smallest * rec / 640 < 24:
        rec += 160
    print(f"Small-object classes: {small}")
    print(f"Smallest median side at 640 px: {smallest:.1f} px")
    print(f"\n--> Recommend imgsz={rec}. Raising imgsz is the lever that works for")
    print("    small objects; a bigger model at 640 is not. Cost is roughly")
    print(f"    (imgsz/640)^2 in time and memory, so {rec} costs about "
          f"{(rec/640)**2:.1f}x a 640 run.")
    CFG["imgsz"] = rec
else:
    print("No small-object classes. imgsz=640 is fine.")

# Re-derive batch for the new resolution. Activation memory scales with pixel
# count, so raising imgsz while leaving batch alone is a silent OOM waiting to
# happen - it was a real defect in the first version of this notebook.
REC = recommend(FACTS, imgsz=CFG["imgsz"], base_imgsz=640)
CFG["batch"], CFG["workers"], CFG["cache"] = REC.batch, REC.workers, REC.cache

print(f"\nimgsz {CFG['imgsz']}  batch {CFG['batch']}  "
      f"workers {CFG['workers']}  cache {CFG['cache']!r}")
for n in REC.notes:
    print(f"  - {n}")
for w in REC.warnings:
    print(f"  ! {w}")

In [ ]:
# --- eyeball the labels. Always. -----------------------------------------
import cv2, random
random.seed(0)
PAL = [BLUE, "#1baf7a", RED, "#4a3aa7", "#eb6834", "#eda100", "#e34948"]

stems = random.sample(list(GT["train"]), min(6, len(GT["train"])))
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, stem in zip(axes.ravel(), stems):
    hits = list((CFG["data_root"] / "images" / "train").glob(stem + ".*"))
    if not hits:
        ax.axis("off"); continue
    img = cv2.cvtColor(cv2.imread(str(hits[0])), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for c, b in GT["train"][stem]:
        x1, y1, x2, y2 = (b * np.array([w, h, w, h])).astype(int)
        col = tuple(int(PAL[c % len(PAL)].lstrip("#")[i:i+2], 16) for i in (0, 2, 4))
        cv2.rectangle(img, (x1, y1), (x2, y2), col, 2)
        cv2.putText(img, CANONICAL_CLASSES[c], (x1, max(12, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, col, 1, cv2.LINE_AA)
    ax.imshow(img); ax.set_title(stem[:38], fontsize=9, color=INK_SOFT)
    ax.axis("off"); ax.grid(False)
plt.suptitle("Sample training labels - check these look right before spending GPU hours",
             x=0.02, ha="left", fontsize=12, color=INK)
plt.tight_layout(); plt.show()

## 5 · Train the detector

**Run this from a terminal, not from a notebook cell.**

The first run of this project died here, and the Jupyter log explains why: the
training process starved the VM, the websocket timed out for 118 seconds,
autosave then failed with `OSError: [Errno 12] Cannot allocate memory`, and when
the OOM killer fired it took the kernel and every cell output with it. A detached
process writing to a log file survives a dropped tab, a closed laptop and a
restarted Jupyter — and its output is still there afterwards to read.

```bash
# inside the container
cd /workspace
mkdir -p runs

nohup python src/train_ppe.py \
    --data  data/ppe_yolo26/data.yaml \
    --imgsz 960 \
    --epochs 120 \
    > runs/train.log 2>&1 &

tail -f runs/train.log
```

`train_ppe.py` probes memory, derives `batch`, `workers` and `cache` from what the
machine actually has, refuses to start if the budget is hopeless, and writes
`memory_plan.json` next to the run so you can see afterwards what it chose. Check
the plan without training anything:

```bash
python src/train_ppe.py --data data/ppe_yolo26/data.yaml --imgsz 960 --dry-run
```

An interrupted run resumes:

```bash
python src/train_ppe.py --resume runs/ppe/detect_v1/weights/last.pt
```

The pose model stays stock — COCO keypoints generalise well and there is no
keypoint-annotated PPE data to fine-tune on. Only the detector is trained.

**Augmentation choices, and why.** Site cameras are fixed, level and often
distant, so aggressive rotation teaches poses that never occur. Scale jitter and
mosaic do the heavy lifting for small objects; HSV covers the difference between
overcast and direct sun; horizontal flip is free. `close_mosaic` turns mosaic off
for the last stretch so the model finishes on undistorted images.

**Class imbalance.** Ultralytics has no per-class loss weighting, so the balance
was already handled upstream by oversampling violation-bearing images in the
train split. Nothing more to do here — and nothing was oversampled in val or
test, which is what keeps the recall figures in section 6 honest.

**Batch size and the learning rate.** Ultralytics accumulates gradients to a
nominal batch of 64, so a small `batch` does not change the effective batch size
or the LR schedule — only wall-clock per epoch. Lowering `batch` to fit memory
costs you time, not accuracy.

In [ ]:
# Preflight. Cheap to run, and it turns a four-hour failure into a ten-second one.
FACTS = probe()
REC   = recommend(FACTS, imgsz=CFG["imgsz"], base_imgsz=640)
print(report(FACTS, REC))

ok, msg = preflight(FACTS, REC)
print(f"\npreflight: {msg}")

TRAIN_CMD = (f"nohup python src/train_ppe.py --data {DATA_YAML} "
             f"--imgsz {CFG['imgsz']} --epochs {CFG['epochs']} "
             f"> runs/train.log 2>&1 &")
print("\nRun this in a terminal inside the container:\n")
print(f"  {TRAIN_CMD}")
print("  tail -f runs/train.log")

### If you would rather train in the kernel anyway

For a short smoke test — a few hundred images and five epochs — running in the
notebook is fine and the feedback is quicker. For a real run it is not worth the
risk. If you do use the cell below, close every other notebook first: each idle
kernel holds its own copy of torch and CUDA context.

In [ ]:
RUN_IN_KERNEL = False    # set True only for a short smoke test

if not RUN_IN_KERNEL:
    print("Skipped - train with src/train_ppe.py instead (see the cell above).")
    BEST = Path(CFG["run_project"]) / CFG["run_name"] / "weights" / "best.pt"
    print(f"Expecting weights at: {BEST}")
else:
    model = YOLO(CFG["det_weights"])
    results = model.train(
        data       = str(DATA_YAML),
        epochs     = CFG["epochs"],
        imgsz      = CFG["imgsz"],
        batch      = CFG["batch"],
        workers    = CFG["workers"],
        cache      = CFG["cache"],
        device     = CFG["device"],
        patience   = CFG["patience"],
        project    = CFG["run_project"],
        name       = CFG["run_name"],
        exist_ok   = True,

        optimizer  = "auto",   # YOLO26 resolves this to MuSGD
        cos_lr     = True,
        warmup_epochs = 3.0,
        amp        = True,
        seed       = 0,
        plots      = True,

        degrees = 3.0, translate = 0.10, scale = 0.55, shear = 1.0,
        perspective = 0.0004, fliplr = 0.5, flipud = 0.0,
        hsv_h = 0.015, hsv_s = 0.7, hsv_v = 0.45,
        mosaic = 1.0, close_mosaic = 15, mixup = 0.05,
    )
    BEST = Path(results.save_dir) / "weights" / "best.pt"

print(f"\nbest weights: {BEST}")

## 6 · Evaluate — negative-class recall at a real operating point

mAP is the right number for comparing architectures and the wrong number for
deciding whether to deploy a safety report. The question here is narrower:

> at the confidence threshold we will actually run, what fraction of genuine
> `no-helmet` instances do we catch, and how often do we accuse a compliant
> worker?

That is per-class recall and precision at one threshold. So: one inference pass
at a low floor, then sweep thresholds over the cached detections and choose the
operating point per class, deliberately.

In [ ]:
best = YOLO(str(BEST))
val_metrics = best.val(data=str(DATA_YAML), split="val", imgsz=CFG["imgsz"],
                       device=CFG["device"], plots=True, verbose=False)

per_class = pd.DataFrame({
    "class":    [CANONICAL_CLASSES[int(c)] for c in val_metrics.box.ap_class_index],
    "mAP50":    val_metrics.box.ap50,
    "mAP50-95": val_metrics.box.maps[val_metrics.box.ap_class_index]
                if hasattr(val_metrics.box, "maps") else np.nan,
    "precision": val_metrics.box.p,
    "recall":    val_metrics.box.r,
}).set_index("class")
per_class["violation"] = [c in VIOLATION_CLASSES for c in per_class.index]
display(per_class.round(3))
print(f"\noverall mAP50 {val_metrics.box.map50:.3f}   mAP50-95 {val_metrics.box.map:.3f}")
print("Useful for tracking progress between runs. Not the number to deploy on.")

In [ ]:
from ppe_eval import collect_predictions, sweep, pick_operating_point

PRED = collect_predictions(best, CFG["data_root"] / "images" / "val",
                           conf_floor=0.05, imgsz=CFG["imgsz"],
                           end2end=CFG["end2end"], device=CFG["device"])
SW = sweep(PRED, GT["val"], CANONICAL_CLASSES,
           thresholds=np.round(np.arange(0.05, 0.96, 0.05), 2))
print(f"swept {SW['threshold'].nunique()} thresholds over "
      f"{sum(len(v) for v in PRED.values()):,} cached detections")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.2), sharey=True)
for ax, (cls, colour) in zip(axes, zip(VIOLATION_CLASSES, SERIES3)):
    d = SW[SW["class"] == cls].sort_values("threshold")
    ax.plot(d["threshold"], d["recall"],    color=colour, lw=2, label="recall")
    ax.plot(d["threshold"], d["precision"], color=colour, lw=2, ls="--",
            label="precision")
    floor = CFG["precision_floor"][cls]
    ax.axhline(floor, color=INK_SOFT, lw=1, ls=":")
    ax.text(0.97, floor + 0.02, f"precision floor {floor:.2f}", ha="right",
            fontsize=8.5, color=INK_SOFT)
    op = pick_operating_point(SW, cls, floor)
    if op:
        ax.axvline(op["threshold"], color=INK_SOFT, lw=1)
        ax.scatter([op["threshold"]], [op["recall"]], s=46, color=colour,
                   zorder=5, edgecolor=SURFACE, linewidth=2)
        ax.annotate(f"conf {op['threshold']:.2f}\nrecall {op['recall']:.2f}",
                    (op["threshold"], op["recall"]), textcoords="offset points",
                    xytext=(8, -26), fontsize=9, color=INK)
    ax.set_title(cls); ax.set_xlabel("confidence threshold")
    ax.set_ylim(0, 1.05); ax.set_xlim(0.05, 0.95)
axes[0].set_ylabel("precision / recall")
axes[0].legend(loc="lower left")
plt.suptitle("Solid = recall, dashed = precision. Marker = chosen operating point.",
             x=0.02, ha="left", fontsize=11, color=INK_SOFT)
plt.tight_layout(); plt.show()

In [ ]:
# --- the decision table --------------------------------------------------
rows = []
for cls in VIOLATION_CLASSES:
    floor = CFG["precision_floor"][cls]
    op = pick_operating_point(SW, cls, floor)
    if op is None:
        rows.append({"class": cls, "precision_floor": floor, "threshold": None,
                     "precision": None, "recall": None, "support": None,
                     "verdict": "NOT FIT TO REPORT - no threshold clears the floor"})
    else:
        rows.append({"class": cls, "precision_floor": floor,
                     "threshold": op["threshold"], "precision": round(op["precision"], 3),
                     "recall": round(op["recall"], 3), "support": op["support"],
                     "verdict": ("deployable" if op["recall"] >= 0.70 else
                                 "advisory only - misses too many")})
OP = pd.DataFrame(rows).set_index("class")
display(OP)

def _thr(c, default=0.50):
    # None becomes NaN once pandas types the column, and NaN is truthy, so an
    # `or` guard here would silently propagate NaN into the inference config.
    v = OP.loc[c, "threshold"]
    return float(v) if v is not None and v == v else default

CONF_BY_CLASS = {c: _thr(c) for c in VIOLATION_CLASSES}
print("\nPer-class confidence thresholds for deployment:")
print(json.dumps({k: float(v) for k, v in CONF_BY_CLASS.items()}, indent=2))
print('''
Read the verdict column literally. A class with no threshold that clears its
precision floor is not ready to report on, and saying so is the finding. Quietly
lowering the floor to make the table look better is how a safety system loses the
confidence of the people who have to act on it.''')

## 7 · Pose model

Stock weights, no training. A quick check that keypoints land where they should
before wiring the fusion layer to them.

In [ ]:
pose = YOLO(CFG["pose_weights"])
sample = next(iter((CFG["data_root"] / "images" / "val").glob("*.jp*g")))
pr = pose.predict(str(sample), imgsz=CFG["imgsz"], device=CFG["device"], verbose=False)[0]

n = 0 if pr.keypoints is None else len(pr.keypoints.data)
print(f"{sample.name}: {n} skeletons")
fig, ax = plt.subplots(figsize=(7.5, 5.5))
ax.imshow(cv2.cvtColor(pr.plot(), cv2.COLOR_BGR2RGB)); ax.axis("off"); ax.grid(False)
ax.set_title("Stock pose model output", color=INK)
plt.tight_layout(); plt.show()

## 8 · Fusion — per-person compliance

Each PPE box is assigned to the nearest eligible person, then tested against the
body region it claims to protect:

| item | anatomical test |
|---|---|
| helmet | box centre within `0.42 × torso` of the head anchor (mean of visible nose/eyes/ears), centre above the nose, top edge above the eye line |
| vest | box covers ≥ 30 % of the torso quad built from shoulders and hips |
| boots | one box per ankle, **claimed exclusively** — a standing worker's ankles are closer together than the search radius, so without exclusive claiming a single boot would satisfy both feet |

Then detector and pose evidence are reconciled:

| detector says | pose says | verdict |
|---|---|---|
| `no-helmet` | nothing on the head | **VIOLATION** — corroborated |
| `helmet` | not on the head | **REVIEW** — carried, hanging, or someone else's |
| `helmet` + `no-helmet` | on the head | **REVIEW** — detector probably wrong |
| `helmet` + `no-helmet` | not on the head | **VIOLATION** — has it, is not wearing it |
| `helmet` | on the head | COMPLIANT |
| anything | region not observable | **INDETERMINATE** — never a violation |

That last row is the one that earns its keep. A worker whose legs are out of
frame has no observable feet, and reporting a footwear violation there is a false
accusation. One false accusation costs more trust than ten missed detections.

In [ ]:
from ppe_fusion import (FusionConfig, Verdict, assess_frame_ultralytics,
                        summarise, compliance_rate, annotate, ITEMS)

FUSE = FusionConfig(
    det_conf_ppe = 0.35,
    det_conf_neg = float(np.mean([v for v in CONF_BY_CLASS.values() if v])),
    kp_conf      = 0.35,
    require_both_feet = True,
)
print(f"negative-class confidence floor from section 6: {FUSE.det_conf_neg:.2f}")

def assess(path):
    dr = best.predict(str(path), imgsz=CFG["imgsz"], conf=0.05,
                      device=CFG["device"], verbose=False)[0]
    pr = pose.predict(str(path), imgsz=CFG["imgsz"], conf=0.30,
                      device=CFG["device"], verbose=False)[0]
    return dr, pr, assess_frame_ultralytics(dr, pr, CANONICAL_CLASSES, FUSE)

dr, pr, people = assess(sample)
for a in people:
    print(f"\nperson #{a.person_id}  torso={a.torso_scale:.0f}px  truncated={a.truncated}")
    for item in ITEMS:
        f = a.findings[item]
        print(f"   {item:<7} {f.verdict.value:<32} conf={f.confidence:.2f} "
              f"geo={f.geometry_score:.2f}")
        print(f"           {f.note}")

In [ ]:
frame = cv2.imread(str(sample))
fig, axes = plt.subplots(1, 2, figsize=(15, 5.6))
axes[0].imshow(cv2.cvtColor(dr.plot(), cv2.COLOR_BGR2RGB))
axes[0].set_title("Raw detector output - boxes, no owner", color=INK)
axes[1].imshow(cv2.cvtColor(annotate(frame, people), cv2.COLOR_BGR2RGB))
axes[1].set_title("After fusion - verdicts attached to people", color=INK)
for ax in axes:
    ax.axis("off"); ax.grid(False)
plt.tight_layout(); plt.show()

In [ ]:
# --- batch over the test split ------------------------------------------
from collections import Counter

test_imgs = sorted((CFG["data_root"] / "images" / "test").glob("*.jp*g"))[:300]
rows, tally = [], Counter()
for p in test_imgs:
    _, _, ppl = assess(p)
    for a in ppl:
        r = a.to_row(); r["image"] = p.name
        rows.append(r)
        for item, f in a.findings.items():
            tally[(item, f.verdict.value)] += 1

REPORT = pd.DataFrame(rows)
out_csv = Path(CFG["run_project"]) / CFG["run_name"] / "compliance_report.csv"
out_csv.parent.mkdir(parents=True, exist_ok=True)
REPORT.to_csv(out_csv, index=False)
print(f"{len(REPORT):,} person-records from {len(test_imgs):,} images -> {out_csv}")
display(REPORT.head(10))

print("\nCompliance among people where the item was observable:")
obs = {}
for item in ITEMS:
    col = REPORT[f"{item}_verdict"]
    dec = col[col != Verdict.INDETERMINATE.value]
    obs[item] = (len(dec), (dec == Verdict.COMPLIANT.value).mean() if len(dec) else float("nan"))
    print(f"  {item:<7} {obs[item][1]:.1%}   (n={obs[item][0]:,} observable, "
          f"{len(col) - len(dec):,} not observable)")

In [ ]:
# --- verdict distribution. Identity is in the axis labels, not the colour:
# status green and red are not reliably separable under deuteranopia.
def bucket(v):
    if v.startswith("REVIEW"):        return "REVIEW"
    if v.startswith("INDETERMINATE"): return "INDETERMINATE"
    return v

ORDER = ["COMPLIANT", "VIOLATION", "REVIEW", "INDETERMINATE"]
grid = pd.DataFrame(
    [[sum(n for (it, v), n in tally.items() if it == item and bucket(v) == b)
      for b in ORDER] for item in ITEMS],
    index=ITEMS, columns=ORDER)
display(grid)

fig, axes = plt.subplots(1, 3, figsize=(14.5, 3.6), sharex=True)
xmax = grid.values.max() * 1.28 or 1
for ax, item in zip(axes, ITEMS):
    vals = grid.loc[item, ORDER]
    bars = ax.barh(ORDER[::-1], vals[::-1],
                   color=[STATUS[b] for b in ORDER[::-1]], height=0.62)
    bar_labels(ax, bars, pad=xmax * 0.015, horiz=True)
    ax.set_title(item, color=INK); ax.set_xlim(0, xmax)
    ax.grid(axis="y", visible=False)
axes[1].set_xlabel("person-records")
plt.suptitle("Verdict distribution per PPE item, test split",
             x=0.02, ha="left", fontsize=12, color=INK)
plt.tight_layout(); plt.show()

rev = grid["REVIEW"].sum(); tot = grid.values.sum()
print(f"\n{rev:,} of {tot:,} records ({rev/tot:.1%}) need a human look.")
print("That queue is a feature. A system that never says 'I am not sure' is")
print("either very good or hiding its errors, and it is rarely the first.")

## 9 · Video — temporal voting

A single frame is a weak basis for a safety finding. Motion blur, a turned head,
one bad frame of occlusion — any of these flips a verdict. So track people across
frames and only report a violation once it holds over **k of the last n frames**
for that tracked person.

This is the cheapest accuracy improvement available and it costs nothing at
training time.

In [ ]:
from collections import deque, defaultdict

class TemporalVoter:
    '''Confirms a verdict only when it holds for k of the last n frames per track.

    INDETERMINATE frames are recorded but never vote, so a worker who walks
    behind a stack of materials does not accumulate false violations while
    obscured - the window simply has less to say about them.'''

    def __init__(self, n=15, k=9):
        self.n, self.k = n, k
        self.hist = defaultdict(lambda: defaultdict(lambda: deque(maxlen=n)))

    def update(self, track_id, findings):
        for item, f in findings.items():
            self.hist[track_id][item].append(f.verdict.value)

    def confirmed(self, track_id):
        out = {}
        for item, dq in self.hist[track_id].items():
            votes = [v for v in dq if not v.startswith("INDETERMINATE")]
            if len(votes) < self.k:
                out[item] = ("INDETERMINATE_insufficient_evidence", 0.0)
                continue
            top, cnt = Counter(votes).most_common(1)[0]
            out[item] = (top, cnt / len(votes)) if cnt >= self.k else \
                        ("INDETERMINATE_unstable", cnt / len(votes))
        return out


def process_video(src, out_path=None, max_frames=None, stride=1):
    cap = cv2.VideoCapture(str(src))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = (cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
              if out_path else None)

    voter, log, i = TemporalVoter(n=15, k=9), [], 0
    try:
        while True:
            ok, frame = cap.read()
            if not ok or (max_frames and i >= max_frames):
                break
            i += 1
            if i % stride:
                continue

            # Track on the pose model so person identity is stable across frames.
            pr = pose.track(frame, persist=True, imgsz=CFG["imgsz"],
                            conf=0.30, device=CFG["device"], verbose=False)[0]
            dr = best.predict(frame, imgsz=CFG["imgsz"], conf=0.05,
                              device=CFG["device"], verbose=False)[0]
            ppl = assess_frame_ultralytics(dr, pr, CANONICAL_CLASSES, FUSE)

            ids = (pr.boxes.id.int().cpu().tolist()
                   if pr.boxes is not None and pr.boxes.id is not None
                   else list(range(len(ppl))))
            for a in ppl:
                tid = ids[a.person_id] if a.person_id < len(ids) else a.person_id
                voter.update(tid, a.findings)
                conf = voter.confirmed(tid)
                log.append({"frame": i, "track": tid,
                            **{f"{k_}_confirmed": v[0] for k_, v in conf.items()},
                            **{f"{k_}_agreement": round(v[1], 2) for k_, v in conf.items()}})
            if writer:
                writer.write(annotate(frame, ppl))
    finally:
        cap.release()
        if writer:
            writer.release()
    return pd.DataFrame(log)

# VID = Path("/workspace/samples/site_camera.mp4")
# LOG = process_video(VID, out_path="runs/ppe/annotated.mp4", max_frames=600, stride=2)
# confirmed = LOG.groupby("track").last()
# display(confirmed)
print("Uncomment the three lines above and point VID at a site clip.")

## 10 · Export

```python
best.export(format="onnx",   imgsz=CFG["imgsz"], opset=17, simplify=True, dynamic=False)
best.export(format="engine", imgsz=CFG["imgsz"], half=True)   # TensorRT, on the target GPU
```

Export the pose model too — the fusion layer needs both at inference. Build
TensorRT engines on the machine that will run them; they are not portable across
GPU architectures or driver versions.

YOLO26's NMS-free head is what makes this worth doing: with no NMS to reimplement
in the serving path, the exported graph is the whole pipeline. Keep `end2end=True`
consistent between the threshold sweep in section 6 and deployment, or the
operating points you chose will not mean the same thing.

## 11 · Fine-tune on your own footage — do not skip this

Everything above is trained on stock and web photography: sharp, well lit, shot
from a few metres away. Site CCTV is none of those things. The domain gap is the
single largest source of error in a system like this, and it is larger than the
gap between model sizes.

1. Pull 300–500 frames from your own cameras, spread across weather, time of day
   and camera position. Deliberately include the awkward cases — workers at
   distance, partial occlusion, high-vis in low sun.
2. Label them with the same seven classes. Consistency with the taxonomy matters
   more than volume.
3. Fine-tune from `best.pt` at a low learning rate:

```python
tuned = YOLO(str(BEST))
tuned.train(data="site_data.yaml", epochs=40, imgsz=CFG["imgsz"],
            lr0=0.001, warmup_epochs=1.0, freeze=10, device=CFG["device"])
```

4. Re-run section 6 on a test split drawn **only** from your own footage. That
   number is the one to quote. Nothing derived from the public datasets is.

## What to check before this goes anywhere near a client

- **Near-duplicate leakage.** Several public sources are video-derived. The
  harvester deduplicates by difference hash and keeps near-duplicates in the same
  split, but visually similar frames can still straddle train and val, so section
  6's figures are mildly optimistic. Hold back a genuinely independent set.
- **Footwear is the weak leg.** Negative footwear labels come from essentially one
  public source. Expect `no-boots` recall to trail helmet and vest, and treat
  footwear findings as advisory until site data is in.
- **Licences.** Check `ATTRIBUTION.txt`. If `include_nc=True` was ever set, the
  weights carry SH17's non-commercial restriction and must not be used in
  commercial delivery work.
- **What the model cannot know.** A vest can be unfastened, a chinstrap undone, a
  hard hat past its expiry date. This detects presence and position, not fitness
  for purpose or correct use.
- **People, not just pixels.** Worker-monitoring imagery carries data-protection
  obligations — lawful basis, retention, notice, and consultation before it goes
  live. Worth settling early rather than after the first report lands in someone's
  inbox.